# Oracle AI Agent Memory — End-to-End Support Copilot

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/agent_memory/oamp_support_assistant_example.ipynb) [![Oracle AI Agent Memory docs](https://img.shields.io/badge/Docs-Oracle%20AI%20Agent%20Memory%2026.4-C74634?logo=oracle&logoColor=white)](https://docs.oracle.com/en/database/oracle/agent-memory/26.4/)

This notebook teaches how to build a continuous customer-support copilot with Oracle AI Agent Memory and the OpenAI Agents SDK. It follows one damaged-delivery case from setup through knowledge ingestion, agent tool use, conversation compaction, correction, isolation, retention, and shutdown.

The goal is to show how memory changes an agent from a stateless question-answering component into an application that can preserve durable context, retrieve it safely, and keep that context correct over time. Credentials are read from environment variables and are never stored in this file.

## Capability map

| Capability | Why this is needed | How this notebook demonstrates it |
|---|---|---|
| Background extraction workflows | Agent responses should not wait for every memory-extraction task to finish | Raw messages return promptly while extraction runs in an in-process worker |
| Semantic & hybrid search | Business queries mix exact identifiers with natural-language descriptions | Vector search retrieves the policy from an exact code (<code>RET-14D</code>) or a paraphrase; the in-database route adds keyword fusion |
| Custom extraction instructions | Generic extraction may preserve noise or miss domain-critical facts | The extractor keeps durable support facts and ignores greetings, secrets, and transient wording |
| Context cards | Long transcripts increase cost and distract the agent from current work | A prompt-ready context card replaces replaying the full Agents SDK transcript |
| Metadata filtering and inheritance | Semantic relevance alone cannot enforce tenant, workflow, or approval boundaries | Tenant, case, channel, tags, and approval state flow from messages to extracted memories |
| Update APIs | Real-world facts and user preferences change, so append-only memory can become contradictory | The agent corrects a communication preference; the app updates thread configuration and policy chunks |
| Pluggable embedding route | Some teams want a hosted model; others need embedding execution and governance inside the database | Default OpenAI <code>text-embedding-3-small</code>, or an in-database ONNX model via a single switch |
| Chunked semantic indexing | Large documents exceed useful embedding windows and contain several retrieval topics | One logical policy record is indexed as caller-owned chunks and returned as one logical result |
| Exact scope matching | A relevant result is still wrong if it belongs to another user or agent | <code>exact_user_match=True</code> prevents another customer's memory from crossing the boundary |
| TTL and deletion | Operational and regulated data should not live forever by accident | A short-lived operational memory is created, refreshed, and deleted |
| Typed manual memories | Agents need to distinguish facts, preferences, guidelines, and general recollections | Facts, guidelines, preferences, and general memories retain distinct record types |

## Sources checked

The API shapes in this notebook were checked against the Oracle AI Agent Memory package and its [documentation](https://docs.oracle.com/en/database/oracle/agent-memory/26.4/). The Agents SDK pattern follows the official [running agents](https://openai.github.io/openai-agents-python/running_agents/), [function tools](https://openai.github.io/openai-agents-python/tools/), and [results](https://openai.github.io/openai-agents-python/results/) documentation.


# Part 1 — Use Case and Setup

## Use Case

An online electronics retailer is building a support copilot. A customer reports that <code>ORDER-7421</code> arrived with a damaged screen under case <code>CASE-1042</code>. The copilot must determine whether policy <code>RET-14D</code> applies, remember the customer's communication preference, and continue helping when the customer returns later.

A single response is not enough. The application needs memory because the useful context is distributed across time and data types:

1. **Conversation history:** what the customer and agent said in earlier turns.
2. **Durable facts:** the damaged order, confirmed case state, and approved replacement.
3. **Preferences:** whether the customer wants SMS or email updates.
4. **Guidelines:** the return policy and escalation procedure.
5. **Retrieval boundaries:** tenant, user, agent, case, and approval state.
6. **Lifecycle changes:** preferences can be corrected, policies can be revised, and temporary operational notes should expire.

Without durable memory, every request would need the full transcript and policy set, the agent could forget corrections, and a semantically relevant record from another customer could be returned accidentally. Oracle AI Agent Memory gives the application a durable system of record, while the OpenAI agent uses narrow tools to retrieve and update only the context needed for the current task.

### Workflow

~~~mermaid
sequenceDiagram
    actor Customer
    participant App as Support application
    participant Agent as OpenAI agent
    participant Memory as Oracle AI Agent Memory
    participant DB as Oracle AI Database

    Customer->>App: Report damaged order
    App->>Memory: Persist message with case metadata
    Memory->>DB: Store raw message and retrieval data
    Memory-->>App: Return after durable write
    Memory->>Memory: Extract durable facts in background
    App->>Memory: Request context card
    Memory->>DB: Scoped vector search within user and metadata scope
    DB-->>Memory: Policy, facts, and preferences
    Memory-->>App: Compact prompt-ready context
    App->>Agent: Current request plus context card
    Agent->>Memory: Search, remember, or update through tools
    Agent-->>Customer: Grounded support answer
~~~

The rest of the notebook implements this flow in five logical parts: setup, memory infrastructure, knowledge and tools, the continuous workflow, and lifecycle operations.


## Install the Python packages

Install the packages from PyPI. The `[litellm]` extra lets Oracle AI Agent Memory call hosted
embedding and chat models (such as OpenAI) through one interface, which is what the default
embedding route below uses. `openai-agents` provides the agent runtime and `oracledb` is the
database driver.

Run this once in a clean environment. A dedicated virtual environment is recommended so the
database driver, the Agents SDK, and the memory package can be upgraded and tested together.

In [ ]:
%pip install --upgrade pip
%pip install --upgrade "oracleagentmemory[litellm]" openai-agents oracledb

## Standalone Oracle Database setup with Docker

This standalone setup uses the Gerald Venzl <code>gvenzl/oracle-free:23-slim</code> image. Use it for development, workshops, and reproducible local testing. Production deployments should use an appropriately managed Oracle AI Database service and organizational security controls.

### 1. Pull the image and create persistent storage

~~~bash
export ORACLE_PASSWORD='<choose-a-strong-system-password>'
docker pull gvenzl/oracle-free:23-slim
docker volume create oracle-free-data
~~~

### 2. Start Oracle Database

~~~bash
docker run -d \
  --name oracle-free \
  -p 1521:1521 \
  -e ORACLE_PASSWORD="$ORACLE_PASSWORD" \
  -v oracle-free-data:/opt/oracle/oradata \
  gvenzl/oracle-free:23-slim
~~~

Follow startup until the log reports that the database is ready:

~~~bash
docker logs -f oracle-free
~~~

### 3. Create a dedicated application user

Connect to the default pluggable database:

~~~bash
docker exec -it oracle-free sqlplus system/"$ORACLE_PASSWORD"@FREEPDB1
~~~

Then run the following SQL, replacing the placeholder password:

~~~sql
CREATE USER OAMP_APP IDENTIFIED BY "<choose-a-strong-app-password>";
GRANT CREATE SESSION, CREATE TABLE, CREATE SEQUENCE, CREATE VIEW,
      CREATE PROCEDURE, CREATE JOB TO OAMP_APP;
GRANT UNLIMITED TABLESPACE TO OAMP_APP;
~~~

The application user also needs access to a database-resident embedding model. Import or provision the model according to your database administration process, then set <code>INDB_EMBED_MODEL</code> to its Oracle identifier.

### 4. Export notebook configuration

~~~bash
export DB_USER='OAMP_APP'
export DB_PASSWORD='<your-app-password>'
export DB_CONNECT_STRING='localhost:1521/FREEPDB1'
export INDB_EMBED_MODEL='ALL_MINILM_L12_V2'
export INDB_EMBED_DIM='384'
export OPENAI_API_KEY='<your-provider-key>'
~~~

### 5. Stop or restart the standalone database

~~~bash
docker stop oracle-free
docker start oracle-free
~~~

The named volume preserves database files across container restarts. Remove the container and volume only when you intentionally want to delete the local database state.


## Validate the runtime and required credentials

Oracle AI Agent Memory supports Python 3.10 through 3.13. Use a kernel in that range and set these variables before launching Jupyter:

1. <code>OPENAI_API_KEY</code>: used by both the Oracle Agent Memory LLM adapter and the OpenAI Agents SDK.
2. <code>DB_USER</code>, <code>DB_PASSWORD</code>, and <code>DB_CONNECT_STRING</code>: Oracle Database credentials and DSN.
3. <code>INDB_EMBED_MODEL</code>: optional DB-resident ONNX model name; the example defaults to <code>ALL_MINILM_L12_V2</code>.
4. <code>INDB_EMBED_DIM</code>: optional vector dimension; the example defaults to <code>384</code>.
5. <code>INDB_EMBED_MAX_TOKENS</code>: optional documented input budget for that model; the example uses <code>512</code>.

Failing early is important because a partially configured notebook can otherwise create database state before the missing dependency is discovered.


In [ ]:
import os
import sys

required_env = ["OPENAI_API_KEY", "DB_USER", "DB_PASSWORD", "DB_CONNECT_STRING"]
missing_env = [name for name in required_env if not os.environ.get(name)]

if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError("Use a supported Python 3.10 through 3.13 kernel.")
if missing_env:
    raise EnvironmentError(f"Missing required environment variables: {missing_env}")

print("Runtime and required environment variables are ready.")


## Verify the installed packages without loading feature classes

This cell imports only the top-level <code>oracleagentmemory</code> module and Python's distribution-version helper. Feature classes are intentionally imported later beside their first use so developers can see which module owns each API.

- <code>oracleagentmemory.__version__</code> reports the installed package version.
- <code>importlib.metadata.version(distribution_name)</code> accepts the installed distribution name and returns its version string.
- Methods such as <code>search_async()</code> and <code>add_messages_async()</code> are instance methods, so they are not imported separately; they become available after their owning class is instantiated.

This progressive-import style is educational rather than mandatory. Python safely permits a consolidated import section in production applications when that better matches project conventions.


In [ ]:
from importlib.metadata import version

import oracleagentmemory

print("oracleagentmemory:", oracleagentmemory.__version__)
print("openai-agents:", version("openai-agents"))


## Create a concurrent Oracle connection pool

The support application performs foreground writes, background extraction, context-card searches, and agent-triggered retrieval. A connection pool lets those operations borrow separate database sessions instead of serializing all work behind one raw connection. This matters most in multi-user services and in any workflow that enables background extraction or per-type context-card searches.

A pool is preferable to a raw connection here because background extraction and context-card per-type searches may overlap.

The next cell introduces <code>import oracledb</code> only when the database connection is needed. <code>oracledb.create_pool()</code> is a module function rather than an Oracle AI Agent Memory class.

### <code>oracledb.create_pool()</code> arguments used below

- <code>user</code>, <code>password</code>, <code>dsn</code>: connection identity and target database.
- <code>min</code>: connections opened initially.
- <code>max</code>: hard pool size, which should cover expected concurrent DB work.
- <code>increment</code>: connections added when demand exceeds the current pool size.


In [ ]:
import oracledb

db_pool = oracledb.create_pool(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
    min=1,
    max=8,
    increment=1,
)

print("Oracle connection pool created.")


# Part 2 — Memory Infrastructure

This part configures the services that make memory durable and retrievable: summarization and extraction, database-managed embeddings, background work, retention, the hybrid store, and thread ownership. Keeping these concerns explicit makes the later agent tools small and easier to secure.


## Configure the Oracle Agent Memory LLM

<code>Llm</code> is the Oracle Agent Memory adapter used for extraction and summarization. It is important because raw conversations are verbose and transient: extraction turns them into durable records, while summarization creates compact continuity for later turns. Support, sales, healthcare coordination, and long-running operational agents all benefit from this separation between raw dialogue and reusable memory.

The next cell imports <code>Llm</code> from <code>oracleagentmemory.core.llms</code>. The class implements the package's LLM adapter interface and stores provider defaults used by thread operations.

### Constructor arguments

- <code>model</code>: provider model identifier. This notebook uses <code>gpt-5.4</code>.
- <code>api_key</code>: omitted intentionally so the adapter reads <code>OPENAI_API_KEY</code> from the environment.
- <code>reasoning_effort</code>: <code>"low"</code> reduces latency for structured extraction and concise summaries.
- <code>max_tokens</code>: caps generated output; it does not cap the input extraction prompt.
- <code>temperature</code>: omitted so provider defaults apply.

The memory LLM is distinct from the OpenAI <code>Agent</code> object created later, even though both use the same model.


In [ ]:
from oracleagentmemory.core.llms import Llm

MODEL_NAME = os.environ.get("OAMP_LLM_MODEL", "gpt-5.4")

memory_llm = Llm(
    MODEL_NAME,
    reasoning_effort="low",
    max_tokens=2_000,
)

print("Required memory LLM:", MODEL_NAME)


## Choose an embedding route: OpenAI (default) or in-database

Oracle AI Agent Memory turns text into vectors through an *embedder*, and this notebook supports two
routes. Switch between them with the `OAMP_EMBED_BACKEND` environment variable (`openai` or `indb`);
every cell after this one is identical.

**Route 1 — OpenAI embedding model (default).** Uses `text-embedding-3-small` through LiteLLM. This is
the route most developers reach for first: it only needs `OPENAI_API_KEY`, requires no database-side
model setup, and produces 1536-dimensional vectors. Because the model runs as an external service, the
store uses pure vector search and we declare the vector width up front.

**Route 2 — in-database embedding.** Oracle AI Database runs the embedding model itself (an imported
ONNX model such as `ALL_MINILM_L12_V2`). Prefer this route in production retrieval pipelines:

- **Fewer network hops.** Text is embedded where it is stored. Neither ingestion nor query embedding
  makes a round trip to an external embedding API, so there is one less network dependency on the path.
- **Lower retrieval latency.** The query is encoded *inside* the database on the same call as the vector
  search, removing a per-query API round trip from the hot retrieval path — the difference is most
  visible at high query rates and on tail latency.
- **Data locality and governance.** Text never leaves the database boundary, and the model is
  administered centrally in the database.
- **Hybrid search.** Because the model and the Oracle Text index are co-located, the in-database route
  can fuse vector similarity with keyword matching for exact identifiers (invoice numbers, policy codes).

**Getting the in-database model.** Import a pretrained ONNX embedding model once with
`DBMS_VECTOR.LOAD_ONNX_MODEL`. Oracle publishes a ready-to-use, augmented `all-MiniLM-L12-v2` model plus
step-by-step instructions:

- ONNX Pipeline Models — Text Embedding: <https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/onnx-pipeline-models-text-embedding.html>
- Pre-built embedding model for Oracle AI Database (blog): <https://blogs.oracle.com/machinelearning/use-our-prebuilt-onnx-model-now-available-for-embedding-generation-in-oracle-database-23ai>

```sql
-- After downloading all_MiniLM_L12_v2.onnx into a database directory (e.g. DM_DUMP):
BEGIN
  DBMS_VECTOR.LOAD_ONNX_MODEL(
    directory  => 'DM_DUMP',
    file_name  => 'all_MiniLM_L12_v2.onnx',
    model_name => 'ALL_MINILM_L12_V2');
END;
/
```

The next cell reads `OAMP_EMBED_BACKEND` and builds the matching embedder together with the search
strategy the store should use.

In [ ]:
from oracleagentmemory.core.embedders import Embedder, OracleDBEmbedder
from oracleagentmemory.core import SearchStrategy

# "openai" (default) uses a hosted OpenAI embedding model; "indb" runs an ONNX model
# inside Oracle AI Database. See the markdown above for the trade-offs.
EMBED_BACKEND = os.environ.get("OAMP_EMBED_BACKEND", "openai").lower()

if EMBED_BACKEND == "openai":
    EMBED_MODEL = os.environ.get("OAMP_EMBED_MODEL", "text-embedding-3-small")
    EMBED_DIM = int(os.environ.get("OAMP_EMBED_DIM", "1536"))
    embedder = Embedder(
        model=EMBED_MODEL,
        embedding_dimension=EMBED_DIM,
        max_input_tokens=8_191,
        normalize=True,
    )
    # An external model runs outside the database, so retrieval is pure vector search
    # and the store must be told the vector width.
    SEARCH_STRATEGY = SearchStrategy.VECTOR
    VECTOR_DIM = EMBED_DIM
elif EMBED_BACKEND == "indb":
    EMBED_MODEL = os.environ.get("INDB_EMBED_MODEL", "ALL_MINILM_L12_V2")
    EMBED_DIM = int(os.environ.get("INDB_EMBED_DIM", "384"))
    embedder = OracleDBEmbedder(
        connection=db_pool,
        model=EMBED_MODEL,
        input_name=os.environ.get("INDB_EMBED_INPUT", "DATA"),
        embedding_dimension=EMBED_DIM,
        max_input_tokens=int(os.environ.get("INDB_EMBED_MAX_TOKENS", "512")),
        normalize=True,
        batch_size=16,
    )
    # The model and text index live in the database, so the managed hybrid index
    # (vector + keyword) is available and its width is known in-database.
    SEARCH_STRATEGY = SearchStrategy.HYBRID
    VECTOR_DIM = None
else:
    raise ValueError(
        f"OAMP_EMBED_BACKEND must be 'openai' or 'indb', got {EMBED_BACKEND!r}."
    )

print(
    f"Embedding route: {EMBED_BACKEND} | model={EMBED_MODEL} | "
    f"{EMBED_DIM}-dim | search={SEARCH_STRATEGY.name}"
)

### Verify the embedding flow

<code>embed_async(texts, is_query)</code> accepts:

- <code>texts</code>: a list of raw strings, and returns a two-dimensional <code>float32</code> matrix.
- <code>is_query</code>: when true, applies any configured query prefix.

This probe confirms the active embedder — OpenAI or in-database — resolves and returns vectors of the
expected width before any schema or search work begins.

In [ ]:
probe_vectors = await embedder.embed_async(
    ["damaged parcel", "replacement shipment"],
    is_query=True,
)

print("Probe shape:", probe_vectors.shape)
print("Probe dtype:", probe_vectors.dtype)


## Define domain-specific extraction instructions

Custom instructions are appended to the built-in extraction prompt; they do not replace the SDK's schema or core extraction rules. Good instructions describe durable business value and exclusions without trying to dictate provider-specific JSON.

For this support use case, the extractor should:

1. Keep stable case identifiers, order identifiers, confirmed issue facts, commitments, and communication preferences.
2. Classify stable statements as <code>fact</code>, <code>preference</code>, or <code>guideline</code> when appropriate.
3. Ignore greetings, speculative troubleshooting, ephemeral wording, credentials, and payment secrets.
4. Preserve exact identifiers such as <code>CASE-1042</code>, <code>ORDER-7421</code>, and <code>RET-14D</code>.


In [ ]:
EXTRACTION_INSTRUCTIONS = """
Keep only durable customer-support information: confirmed case and order facts,
stable communication preferences, commitments, and reusable support guidelines.
Preserve exact identifiers. Ignore greetings, speculation, credentials, payment
secrets, and transient conversational wording.
""".strip()


## Configure faster background extraction

<code>MemoryExtractionConfig</code> groups extraction settings. Omitted fields inherit from the next broader level. Background mode is important for interactive agents because durable message persistence can complete without putting the full extraction latency on the customer-facing response path. It is especially useful for chat, contact-center, and event-ingestion workloads where write responsiveness matters more than immediate consistency of derived memories.

The next cell introduces three related imports from <code>oracleagentmemory.core</code>:

- <code>MemoryExtractionConfig</code>: grouped constructor for extraction behavior.
- <code>MemoryExtractionMode</code>: enum selecting <code>INLINE</code> or <code>BACKGROUND</code> execution.
- <code>BackgroundExtractionQueueFullBehavior</code>: enum selecting <code>DROP</code>, <code>WAIT_THEN_DROP</code>, or <code>WAIT_THEN_RAISE</code> when background capacity is unavailable.

### <code>MemoryExtractionConfig</code> arguments used

- <code>extract_memories=True</code>: enables automatic extraction and therefore requires an LLM.
- <code>extraction_mode=BACKGROUND</code>: message writes return after raw persistence and queue submission instead of waiting for extraction.
- <code>background_extraction_queue_full_behavior=WAIT_THEN_RAISE</code>: waits for capacity and raises <code>TimeoutError</code> if the queue remains full. The raw write has already succeeded.
- <code>background_extraction_queue_put_timeout_seconds=10</code>: queue-capacity wait limit.
- <code>memory_extraction_frequency=1</code>: runs extraction for each appended batch in this demonstration.
- <code>memory_extraction_window=-1</code>: sends only newly added messages, reducing prompt size and latency.
- <code>memory_extraction_token_limit=6000</code>: input budget for extraction and running-summary prompts; values below one disable the bound.
- <code>enable_context_summary=True</code>: maintains a running summary for extraction continuity.
- <code>context_summary_update_frequency=2</code>: refreshes that running summary after two appended messages.
- <code>memory_extraction_custom_instructions</code>: injects the support policy above.
- <code>memory_extraction_inherit_message_metadata</code>: a key list copies only selected top-level message metadata to derived memories.

A production system should choose queue behavior deliberately: <code>DROP</code> favors write availability, <code>WAIT_THEN_DROP</code> adds bounded backpressure without failing the call, and <code>WAIT_THEN_RAISE</code> makes queue saturation visible.


In [ ]:
from oracleagentmemory.core import (
    BackgroundExtractionQueueFullBehavior,
    MemoryExtractionConfig,
    MemoryExtractionMode,
)

extraction_config = MemoryExtractionConfig(
    extract_memories=True,
    extraction_mode=MemoryExtractionMode.BACKGROUND,
    background_extraction_queue_full_behavior=(
        BackgroundExtractionQueueFullBehavior.WAIT_THEN_RAISE
    ),
    background_extraction_queue_put_timeout_seconds=10.0,
    memory_extraction_frequency=1,
    memory_extraction_window=-1,
    memory_extraction_token_limit=6_000,
    enable_context_summary=True,
    context_summary_update_frequency=2,
    memory_extraction_custom_instructions=EXTRACTION_INSTRUCTIONS,
    memory_extraction_inherit_message_metadata=[
        "tenant_id", "case_id", "channel", "tags", "review"
    ],
)


## Define retention defaults

<code>MemoryRetentionConfig</code> controls DB-backed message and memory TTL defaults. Retention is important for privacy, regulatory policy, storage cost, and the quality of retrieval: expired operational details should not continue influencing future answers.

The next cell imports <code>MemoryRetentionConfig</code> from <code>oracleagentmemory.core</code> and <code>TimeToLiveAnchor</code> from <code>oracleagentmemory.core.retention</code>. The config defines schema-level limits; the enum defines the timestamp from which an individual record expires.

### <code>MemoryRetentionConfig</code> constructor arguments

- <code>default_ttl_days</code>: applied when a new write omits <code>ttl_days</code>.
- <code>max_ttl_days</code>: clamps larger explicit TTLs. Passing <code>ttl_days=None</code> uses this maximum when configured.
- <code>TimeToLiveAnchor.CREATED_AT</code>: counts from database creation time.
- <code>TimeToLiveAnchor.TIMESTAMP</code>: counts from the supplied record event timestamp.

Explicit settings make retention behavior independent of installation defaults.


In [ ]:
from oracleagentmemory.core import MemoryRetentionConfig
from oracleagentmemory.core.retention import TimeToLiveAnchor

retention_config = MemoryRetentionConfig(
    default_ttl_days=90,
    max_ttl_days=365,
)


## Create the Oracle AI Database memory store

<code>OracleDBMemoryStore</code> owns the managed records, chunks, schema objects, and retrieval. It adapts
to the embedding route chosen above: the OpenAI route uses vector search, while the in-database route can
use Oracle's managed hybrid index (vector + keyword).

The next cell imports the store plus three configuration enums from <code>oracleagentmemory.core</code>:

- <code>OracleDBMemoryStore</code>: DB-backed store for managed records, chunks, schema objects, and retrieval.
- <code>SchemaPolicy</code>: whether startup validates, creates, upgrades, or recreates managed objects.
- <code>SearchStrategy</code>: vector, keyword, or hybrid retrieval (set for you from the embedding route).
- <code>SearchIndexSyncMode</code>: on-commit, automatic, or manual index refresh (hybrid/keyword only).

### Key constructor arguments

- <code>embedder</code>: the embedder built above (OpenAI or in-database).
- <code>pool</code>: Oracle connection pool.
- <code>schema_policy</code>: <code>CREATE_IF_NECESSARY</code> creates or upgrades managed objects; treat the first hybrid upgrade as a migration.
- <code>memory_store_id</code>: stable managed-object prefix — begins with a letter, letters/digits/underscores only, at most 16 characters.
- <code>search_strategy</code>: <code>VECTOR</code> for external embeddings, <code>HYBRID</code> for the in-database model.
- <code>vector_dim</code>: required for the external (vector) route because the model runs outside the database; omitted for the in-database route.
- <code>search_index_sync=ON_COMMIT</code>: keeps the hybrid/keyword index fresh after each commit; unused by the pure-vector route.
- <code>memory_retention_config</code>: persists explicit retention settings.

For production, switch to <code>REQUIRE_EXISTING</code> during normal startup to validate without DDL.

In [ ]:
from oracleagentmemory.core import (
    OracleDBMemoryStore,
    SchemaPolicy,
    SearchIndexSyncMode,
    SearchStrategy,
)

store_kwargs = dict(
    embedder=embedder,
    pool=db_pool,
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    memory_store_id=os.environ.get("OAMP_MEMORY_STORE_ID", "OAMPDEMO"),
    search_strategy=SEARCH_STRATEGY,
    memory_retention_config=retention_config,
)

if SEARCH_STRATEGY is SearchStrategy.VECTOR:
    # External embeddings: declare the vector width; there is no managed text index to sync.
    store_kwargs["vector_dim"] = VECTOR_DIM
else:
    # In-database hybrid/keyword index: keep it in sync as writes commit.
    store_kwargs["search_index_sync"] = SearchIndexSyncMode.ON_COMMIT

store = OracleDBMemoryStore(**store_kwargs)

print(f"Oracle AI Database memory store is ready ({SEARCH_STRATEGY.name}).")

## Attach the high-level memory component

The next cell imports <code>OracleAgentMemory</code> from <code>oracleagentmemory.core</code>. This is the high-level component used by applications to manage actors, threads, memories, search, and lifecycle operations. Its constructor can accept either a preconfigured <code>store</code> or direct connection/embedder arguments.

- <code>store</code>: reuses the memory store configured above.
- <code>llm</code>: becomes the client-level default for thread extraction and summaries.
- <code>memory_extraction_config</code>: becomes the client-level extraction policy inherited by new threads.

Because an LLM is always present, <code>get_summary()</code> can synthesize a summary instead of using the no-LLM transcript-style behavior.


In [ ]:
from oracleagentmemory.core import OracleAgentMemory

memory = OracleAgentMemory(
    store=store,
    llm=memory_llm,
    memory_extraction_config=extraction_config,
)


## Create actors and a case thread

Each run gets unique IDs while reopening the same managed store.

### Actor methods

- <code>add_user(user_id, information, metadata)</code> stores a searchable user profile.
- <code>add_agent(agent_id, information, metadata)</code> stores a searchable agent profile.

### <code>create_thread()</code> arguments used

- <code>thread_id</code>, <code>user_id</code>, <code>agent_id</code>: durable ownership identifiers.
- <code>metadata</code>: thread-level application data. It is separate from message metadata.
- <code>context_card_token_limit</code>: input budget for the LLM prompt that derives context-card summary and topics.
- <code>context_card_type_search_concurrency</code>: limits per-type context-card retrieval fanout for this live handle; it is not persisted.

The shared <code>CASE_METADATA</code> object is deliberately identical for messages in one extraction pass. Selected inherited metadata must agree across those messages; the SDK raises rather than guessing when values differ.


In [ ]:
from uuid import uuid4

RUN_ID = uuid4().hex[:8]
TENANT_ID = "acme-eu"
USER_ID = f"customer-{RUN_ID}"
AGENT_ID = f"support-agent-{RUN_ID}"
THREAD_ID = f"case-{RUN_ID}"
CASE_ID = "CASE-1042"

CASE_METADATA = {
    "tenant_id": TENANT_ID,
    "case_id": CASE_ID,
    "channel": "web",
    "tags": ["returns", "damaged-delivery"],
    "review": {"status": "approved"},
}


The next cell writes the actor profiles and creates the conversation thread. Profile metadata is useful for filtering and governance, while thread metadata captures the workflow state of this specific case.


In [ ]:
memory.add_user(
    USER_ID,
    "Customer with a damaged-delivery support case.",
    metadata={"tenant_id": TENANT_ID, "segment": "standard"},
)
memory.add_agent(
    AGENT_ID,
    "Support copilot for returns and replacement cases.",
    metadata={"tenant_id": TENANT_ID, "team": "returns"},
)

thread = memory.create_thread(
    thread_id=THREAD_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    metadata={"tenant_id": TENANT_ID, "case_id": CASE_ID, "status": "open"},
    context_card_token_limit=6_000,
    context_card_type_search_concurrency=4,
)

print("Created thread:", thread.thread_id)


# Part 3 — Knowledge Ingestion and Agent Tools

This part loads a policy as chunked knowledge, creates a typed customer preference, and exposes carefully scoped memory operations as OpenAI function tools. The separation matters: ingestion code controls what becomes searchable, while tool wrappers control what the agent is authorized to read or change.


## Prepare a larger policy as caller-owned semantic chunks

Chunked semantic indexing separates the logical content returned to the application from the smaller retrieval rows used by search. It is needed when a policy, contract, case note, or knowledge article covers several topics: embedding the whole document can blur relevance, while returning isolated chunks can lose the complete business artifact. Chunk reconciliation provides focused retrieval without changing what the application stores or returns.

- <code>policy_text</code> is the complete logical record.
- <code>policy_chunks</code> are explicit caller-owned retrieval chunks.
- Supplying <code>list[str]</code> tells the store not to split those chunks again.
- Multiple matching chunks are reconciled back to one logical guideline result.

In a real ingestion pipeline, create chunks along headings or semantic boundaries rather than arbitrary character offsets.


In [ ]:
policy_chunks = [
    (
        "RET-14D eligibility: report transit damage within 14 calendar days "
        "of delivery. Capture the order number and a description of the damage."
    ),
    (
        "Replacement workflow: after eligibility is confirmed, create a "
        "replacement shipment and provide a prepaid return label."
    ),
    (
        "Escalation: if stock is unavailable, offer a refund or an equivalent "
        "replacement and record the customer's choice."
    ),
]

policy_text = "\n\n".join(policy_chunks)
POLICY_ID = f"ret14d-{RUN_ID}"


### Index the policy with <code>store.add_async()</code>

Important arguments:

- <code>contents</code>: list of logical record payloads.
- <code>record_type</code>: one of the supported logical types; this policy is a <code>guideline</code>.
- <code>index_texts</code>: outer list aligns with records; its inner list supplies caller-owned chunks.
- <code>record_ids</code>: stable application-visible ID.
- <code>user_ids</code> and <code>agent_ids</code>: scope the policy to this demo actor pair.
- <code>metadata</code>: structured labels used by filters, not semantic scoring.
- <code>ttl_days</code> and <code>ttl_anchor</code>: explicit lifecycle policy.
- <code>embeddings</code>: omitted, so the configured DB flow derives retrieval vectors/index state.

A string <code>index_texts</code> entry may be chunked by the store; a list entry means the caller owns boundaries.


In [ ]:
policy_ids = await store.add_async(
    contents=[policy_text],
    record_type="guideline",
    index_texts=[policy_chunks],
    record_ids=POLICY_ID,
    user_ids=USER_ID,
    agent_ids=AGENT_ID,
    metadata={
        "tenant_id": TENANT_ID,
        "source": "support-policy",
        "policy_code": "RET-14D",
        "version": 3,
    },
    ttl_days=365,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

print("Indexed policy:", policy_ids)


### Verify logical content remains unchanged

<code>store.get(record_type, record_id)</code> reads the logical record, not individual retrieval chunks.

- <code>record_type</code> selects the managed table family.
- <code>record_id</code> selects the application-visible row.
- The returned record exposes <code>content</code> and decoded <code>metadata</code>.

Search can match any chunk while callers continue to receive the full policy.


In [ ]:
stored_policy = store.get("guideline", POLICY_ID)

print("Logical content preserved:", stored_policy.content == policy_text)
print("Logical characters:", len(stored_policy.content))


## Add a typed manual preference

<code>OracleThread.add_memory_async()</code> scopes a manual memory to the thread by default. Manual writes are important when the application already knows that information is authoritative and should not depend on extraction inference—for example, an explicit consent choice, an operator-approved fact, or a response guideline.

### Arguments

- <code>content</code>: durable text.
- <code>memory_type</code>: <code>memory</code>, <code>fact</code>, <code>guideline</code>, or <code>preference</code>.
- <code>memory_id</code>: optional stable ID, useful for later correction.
- <code>metadata</code>: application-owned labels.
- <code>timestamp</code>: optional event time.
- <code>ttl_days</code> and <code>ttl_anchor</code>: per-record retention.

Typed records help context-card minimums reserve space for preferences and guidelines.


In [ ]:
CONTACT_PREFERENCE_ID = f"contact-pref-{RUN_ID}"

await thread.add_memory_async(
    "The customer prefers SMS for support updates.",
    memory_type="preference",
    memory_id=CONTACT_PREFERENCE_ID,
    metadata=CASE_METADATA,
    ttl_days=180,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

print("Preference ID:", CONTACT_PREFERENCE_ID)


## Wrap Oracle memory operations as OpenAI function tools

The OpenAI Agents SDK derives each tool's name, description, and JSON schema from the Python signature, type annotations, and docstring. Async tools are appropriate because Oracle memory operations and context generation may involve I/O. Narrow tools are important because they keep tenant IDs, metadata policy, TTL, and authorization decisions in application code instead of allowing the model to choose those boundaries.

The official tool documentation recommends:

1. Clear, narrow functions.
2. Typed arguments.
3. Argument descriptions in docstrings.
4. Bounded timeouts for external work.
5. Application-side authorization instead of relying on the model to enforce ownership.

This helper only formats search output; it does not change retrieval.


In [ ]:
def show_hits(hits) -> None:
    for hit in hits:
        print(
            f"- {hit.record.id} [{hit.record.record_type}] "
            f"distance={hit.distance:.4f}: {hit.content}"
        )


### Tool: scoped memory search

The next cell imports <code>function_tool</code> from the OpenAI Agents SDK and <code>json</code> from the standard library. The decorator converts an annotated Python function into a tool schema; <code>json.dumps()</code> converts structured hits into a stable text result for the model.

The decorator's <code>timeout=60</code> bounds one tool invocation. Its default timeout behavior returns an error result to the model rather than leaving the run waiting indefinitely. Function arguments and their docstrings become the tool's input schema.

<code>thread.search_async()</code> combines search relevance, ownership scope, record types, and metadata.

- <code>query</code>: an exact identifier or a natural-language description.
- <code>exact_user_match=True</code>: restricts results to this user's ID.
- <code>exact_agent_match=True</code>: restricts results to this agent's ID.
- <code>exact_thread_match=False</code>: permits durable memories from the same user/agent across threads.
- <code>max_results</code>: upper bound, not a guarantee.
- <code>record_types</code>: limits the logical record categories.
- <code>metadata_filter</code>: applies deterministic AND filtering before relevance ordering.

Metadata is not embedded. It narrows candidates; content still determines semantic ranking.


In [ ]:
import json
from agents import function_tool

@function_tool(timeout=60)
async def search_support_memory(query: str, max_results: int = 5) -> str:
    """Search approved support memories and policies for the current customer.

    Args:
        query: Exact identifier or natural-language retrieval query.
        max_results: Maximum number of logical records to return.
    """
    hits = await thread.search_async(
        query,
        exact_user_match=True,
        exact_agent_match=True,
        exact_thread_match=False,
        max_results=max_results,
        record_types=["memory", "fact", "preference", "guideline"],
        metadata_filter={"tenant_id": TENANT_ID},
    )
    payload = [
        {
            "id": hit.record.id,
            "type": hit.record.record_type,
            "content": hit.content,
            "distance": hit.distance,
        }
        for hit in hits
    ]
    return json.dumps(payload, indent=2)


### Tool: context-card retrieval

<code>get_context_card_async()</code> produces XML-like prompt text containing a thread summary, retrieval topics, relevant durable records, and recent messages.

- <code>fallback_message_count</code>: recent messages used to derive fallback retrieval/summary text.
- <code>max_relevant_results</code>: hard cap on durable records included.
- <code>max_recent_messages</code>: cap on verbatim trailing messages.
- <code>min_relevant_results_by_type</code>: best-effort type minimums for <code>memory</code>, <code>fact</code>, <code>guideline</code>, and <code>preference</code>.

Context-card default retrieval matches the thread's user and agent but intentionally allows related records from other threads for that actor pair.


In [ ]:
@function_tool(timeout=90)
async def get_case_context() -> str:
    """Return compact, retrieval-aware context for the active support case."""
    card = await thread.get_context_card_async(
        fallback_message_count=6,
        max_relevant_results=8,
        max_recent_messages=4,
        min_relevant_results_by_type={
            "preference": 1,
            "guideline": 1,
            "fact": 1,
        },
    )
    return card.content


### Tool: manual typed memory write

This tool allows the agent to explicitly preserve a confirmed fact or preference. The next cell imports <code>Literal</code> from Python's typing module so <code>memory_type</code> becomes a constrained set of values in the generated tool schema.

- <code>content</code>: must be self-contained and durable.
- <code>memory_type</code>: restricted with <code>Literal</code> so the model receives an enum-like tool schema.
- The implementation supplies trusted scope, metadata, and TTL itself rather than allowing the model to choose tenant or user IDs.


In [ ]:
from typing import Literal

@function_tool(timeout=60)
async def remember_customer_fact(
    content: str,
    memory_type: Literal["memory", "fact", "guideline", "preference"] = "fact",
) -> str:
    """Store a confirmed durable support memory.

    Args:
        content: Self-contained fact, preference, guideline, or general memory.
        memory_type: Logical memory category.
    """
    memory_id = await thread.add_memory_async(
        content,
        memory_type=memory_type,
        metadata=CASE_METADATA,
        ttl_days=180,
        ttl_anchor=TimeToLiveAnchor.CREATED_AT,
    )
    return f"Stored {memory_type} memory {memory_id}."


### Tool: thread-scoped memory correction

<code>thread.update_memory_async()</code> only updates memory-like records owned by this exact thread.

- <code>memory_id</code>: immutable record identifier.
- <code>content</code>: replacement text; omitting it preserves content. Passing <code>None</code> is unsupported.
- <code>metadata</code>: whole-object replacement, not a recursive merge. The tool therefore sends the complete desired metadata.
- <code>timestamp</code>: omitted to preserve the event timestamp.
- <code>ttl_days</code>: refreshes expiration.
- <code>ttl_anchor</code>: determines the refresh baseline.

An update reindexes changed content. Authorization remains application code's responsibility.


In [ ]:
@function_tool(timeout=60)
async def update_customer_memory(
    memory_id: str,
    corrected_content: str,
) -> str:
    """Correct a memory in the active support thread.

    Args:
        memory_id: Stable ID of the thread-owned memory.
        corrected_content: Complete replacement content.
    """
    replacement_metadata = {
        **CASE_METADATA,
        "review": {"status": "approved", "updated_by": "support-agent"},
    }
    updated_id = await thread.update_memory_async(
        memory_id,
        content=corrected_content,
        metadata=replacement_metadata,
        ttl_days=180,
        ttl_anchor=TimeToLiveAnchor.CREATED_AT,
    )
    return f"Updated memory {updated_id}."


## Create the OpenAI support agent

The next cell imports <code>Agent</code> and <code>ModelSettings</code> from the OpenAI Agents SDK. <code>Agent</code> describes instructions, tools, and the selected model; <code>ModelSettings</code> carries provider request controls.

### <code>Agent</code> constructor arguments used

- <code>name</code>: traceable logical name.
- <code>model</code>: <code>gpt-5.4</code>.
- <code>instructions</code>: tool policy, grounding requirements, and update safety rules.
- <code>tools</code>: narrow application functions exposed to the model.
- <code>model_settings</code>: provider request controls.

<code>ModelSettings</code> uses low reasoning and verbosity, an 800-token response cap, and <code>parallel_tool_calls=False</code>. Serial tool emission makes mutation order easier to reason about. The Agents SDK runner will call the model again after tool results until it produces final output or reaches <code>max_turns</code>.


In [ ]:
from agents import Agent, ModelSettings

support_agent = Agent(
    name="Acme Returns Copilot",
    model=MODEL_NAME,
    instructions="""
You are a customer-support copilot.
Use search_support_memory for policy or historical claims.
Use remember_customer_fact only for confirmed durable information.
Use update_customer_memory only when the user explicitly corrects a known memory.
Never invent policy terms or expose another user's data.
Answer concisely and mention the policy identifier when one supports the answer.
""".strip(),
    tools=[
        search_support_memory,
        get_case_context,
        remember_customer_fact,
        update_customer_memory,
    ],
    model_settings=ModelSettings(
        reasoning={"effort": "low"},
        verbosity="low",
        max_tokens=800,
        parallel_tool_calls=False,
    ),
)


# Part 4 — Continuous Memory Workflow

This part follows the live case across message persistence, background extraction, filtered retrieval, context-card compaction, agent tool calls, and an explicit preference correction. The important design idea is that the agent does not own durable state: it reads and changes memory through application-controlled interfaces.


## Measure the background extraction request path

The next cell imports <code>Message</code> from <code>oracleagentmemory.apis.thread</code> and <code>time</code> from the standard library. <code>Message</code> provides a typed representation before the thread persists it.

### <code>Message</code> constructor arguments

- <code>role</code>: message author role, such as <code>user</code> or <code>assistant</code>.
- <code>content</code>: raw message text.
- <code>timestamp</code>: optional event timestamp; omitted here so the store uses its normal creation behavior.
- <code>metadata</code>: optional per-message metadata. This example broadcasts metadata through <code>add_messages_async()</code> instead.
- <code>id</code>: optional caller-provided message ID; omission lets the package generate one.

### <code>add_messages_async()</code> arguments

<code>add_messages_async(messages, metadata, ttl_days, ttl_anchor)</code> writes and indexes raw messages.

- <code>messages</code>: typed <code>Message</code> objects or compatible dictionaries.
- <code>metadata</code>: one dictionary broadcasts to the batch; a list must align one-to-one.
- <code>ttl_days</code>: scalar broadcasts or a list aligns per message.
- <code>ttl_anchor</code>: scalar broadcasts or a list aligns per message.
- Extra store-specific keyword arguments are forwarded by the API.

In background mode, successful return means raw persistence succeeded and due extraction was attempted in the worker. It does not mean derived memories are already visible.


In [ ]:
import time
from oracleagentmemory.apis.thread import Message

initial_messages = [
    Message(
        role="user",
        content=(
            "Order ORDER-7421 arrived with a cracked screen. "
            "Please use SMS for updates on CASE-1042."
        ),
    ),
    Message(
        role="assistant",
        content="I will check the damaged-delivery replacement policy.",
    ),
]

started = time.perf_counter()
message_ids = await thread.add_messages_async(
    initial_messages,
    metadata=CASE_METADATA,
)
write_seconds = time.perf_counter() - started

print(f"Raw write returned in {write_seconds:.3f}s")
print("Message IDs:", message_ids)


### Wait at the consistency boundary

<code>wait_for_memory_extraction_async(timeout)</code> waits for earlier background extraction accepted through this same memory component and thread.

- <code>timeout=120</code>: maximum wait in seconds.
- <code>None</code>: wait indefinitely.
- Work started after the wait begins, by another component, or in another process is outside this wait.
- Extraction failures count as finished; production code should combine this boundary with logging and observability.

Use this method before tests, audits, or reads that require extracted memories to be visible. Normal interactive requests can often continue without waiting because the context card still contains recent raw messages.


In [ ]:
await thread.wait_for_memory_extraction_async(timeout=120)
print("Earlier background extraction has finished.")


## Filter inherited metadata and enforce exact scope

The selected message metadata keys should now appear on automatically extracted memories. This is important because similarity is a ranking mechanism, not an authorization mechanism. Multi-tenant assistants, regulated workflows, and human-review pipelines need deterministic filters in addition to semantic relevance. The filter below demonstrates:

1. Scalar exact match for <code>tenant_id</code>.
2. Array membership with <code>$array_contains</code>; a bare list would require exact order and length.
3. Nested mapping match for <code>review.status</code>.
4. Exact user, agent, and thread ownership.
5. Restriction to extracted memory-like record types.

Multiple metadata filter fields use AND semantics. Operators also include <code>$array_contains_any</code> and <code>$not</code>.


In [ ]:
extracted_hits = await thread.search_async(
    "damaged ORDER-7421 and preferred update channel",
    exact_user_match=True,
    exact_agent_match=True,
    exact_thread_match=True,
    max_results=10,
    record_types=["memory", "fact", "preference"],
    metadata_filter={
        "tenant_id": TENANT_ID,
        "tags": {"$array_contains": "damaged-delivery"},
        "review": {"status": "approved"},
    },
)

show_hits(extracted_hits)


## Exercise precision and recall

The same high-level <code>search_async()</code> call handles two query styles that commonly coexist in
business applications: exact business identifiers and paraphrased natural-language descriptions. Vector
search embeds both the stored content and the query, so a semantically close paraphrase and an exact code
can each retrieve the right record.

The example handles:

- An exact identifier query (a policy code such as <code>RET-14D</code>).
- A paraphrased natural-language query about the same policy.

With the in-database route, Oracle additionally fuses keyword matching with vector ranking (hybrid
search), which further sharpens exact-identifier recall. Result order can vary with index configuration
and stored data, so tests assert that the expected IDs are present rather than fixing one exact rank.

In [ ]:
for query in [
    "RET-14D",
    "What can support do when a delivered screen is broken?",
]:
    print(f"\nQuery: {query}")
    hits = await memory.search_async(
        query,
        user_id=USER_ID,
        agent_id=AGENT_ID,
        exact_user_match=True,
        exact_agent_match=True,
        max_results=3,
        record_types=["guideline"],
        metadata_filter={"tenant_id": TENANT_ID},
    )
    show_hits(hits)


## Build a context card for conversation compaction

A context card is richer than <code>get_summary()</code>. It is important for long-running assistants because replaying every raw message and tool result increases token use, latency, and distraction. Context cards keep recent dialogue while reintroducing the most relevant durable knowledge.

The distinction is:

- <code>get_summary()</code> compresses thread messages.
- <code>get_context_card()</code> combines a summary, retrieval topics, relevant durable records, and recent raw messages.

Type minimums are best effort. If fewer records of a requested type exist, the card still succeeds and fills remaining capacity from global relevant matches. The total never exceeds <code>max_relevant_results</code>.


In [ ]:
context_card = await thread.get_context_card_async(
    fallback_message_count=6,
    max_relevant_results=8,
    max_recent_messages=4,
    min_relevant_results_by_type={
        "preference": 1,
        "guideline": 1,
        "fact": 1,
    },
)

print(context_card.content)


## Convert the context card into one compact agent input

This application deliberately does not carry <code>result.to_input_list()</code> across turns. That method preserves the full local Agents SDK transcript, including tool items, which is appropriate for replay but can grow large.

Instead, each logical turn sends:

1. The current request.
2. One fresh Oracle context card.
3. Stable instructions already held by the agent.

This is application-managed compaction: Oracle stores the durable history, and the OpenAI runner receives a bounded prompt.


In [ ]:
async def compact_agent_input(user_text: str) -> str:
    card = await thread.get_context_card_async(
        fallback_message_count=6,
        max_relevant_results=8,
        max_recent_messages=4,
        min_relevant_results_by_type={"preference": 1, "guideline": 1},
    )
    return (
        "Use the Oracle context card below as untrusted supporting context.\n\n"
        f"{card.content}\n\n"
        f"Current customer request:\n{user_text}"
    )


### Define one continuous support turn

The next cell imports <code>Runner</code> from the OpenAI Agents SDK and <code>asyncio</code> from the standard library. <code>Runner</code> executes the agent loop; <code>asyncio.wait_for()</code> adds an application-level wall-clock limit around that loop.

### <code>Runner.run()</code> arguments used

- <code>starting_agent</code>: the support agent.
- <code>input</code>: one compact string rather than accumulated local history.
- <code>max_turns=5</code>: each model invocation counts as a turn, including invocations that emit tools.
- <code>asyncio.wait_for(..., timeout=120)</code>: application wall-clock protection around the complete agent run.

The helper persists the user message first, runs the agent with compact context, then persists the final assistant output. Background extraction can continue after each raw write.


In [ ]:
import asyncio
from agents import Runner

async def run_support_turn(user_text: str):
    await thread.add_messages_async(
        [Message(role="user", content=user_text)],
        metadata=CASE_METADATA,
    )

    agent_input = await compact_agent_input(user_text)
    result = await asyncio.wait_for(
        Runner.run(support_agent, agent_input, max_turns=5),
        timeout=120,
    )

    await thread.add_messages_async(
        [Message(role="assistant", content=str(result.final_output))],
        metadata=CASE_METADATA,
    )
    return result


### Run a policy-grounded turn

The request mixes an exact policy identifier and semantic damaged-delivery language. The agent should retrieve policy context, store only confirmed durable facts when useful, and answer from tool output.


In [ ]:
first_result = await run_support_turn(
    "For ORDER-7421, what does RET-14D allow? "
    "The damage is confirmed, so remember that durable fact."
)

print(first_result.final_output)


### Run an explicit correction through the update tool

The prompt supplies the stable preference memory ID and an explicit correction. This lets the agent safely choose <code>update_customer_memory</code> rather than adding a contradictory second preference.

The update tool replaces the complete metadata object and refreshes the TTL. It cannot change record ownership or memory type.


In [ ]:
second_result = await run_support_turn(
    "Correction: email is now my preferred support channel, not SMS. "
    f"Update memory {CONTACT_PREFERENCE_ID}, then confirm the change."
)

print(second_result.final_output)


### Wait and verify the corrected preference

After the agent turns, wait for earlier automatic extraction so the verification observes a stable point. The explicit update itself is complete when its tool returns, but the two message writes may still have background extraction work.


In [ ]:
await thread.wait_for_memory_extraction_async(timeout=120)

preference_hits = await thread.search_async(
    "preferred support communication channel is email",
    exact_user_match=True,
    exact_agent_match=True,
    exact_thread_match=True,
    max_results=10,
    record_types=["preference"],
)

show_hits(preference_hits)
assert any(hit.record.id == CONTACT_PREFERENCE_ID for hit in preference_hits)


# Part 5 — Lifecycle, Isolation, and Operations

Useful memory must remain correct, appropriately scoped, and removable. This final part updates durable configuration, refreshes and deletes expiring records, proves cross-user isolation, revises chunked knowledge, compares summaries with context cards, and shuts down background work safely.


## Persist a thread update

<code>OracleAgentMemory.update_thread()</code> updates durable thread metadata and runtime configuration. This matters when a case changes status, an extraction policy evolves, or a reopened thread must behave consistently in another process. Without a durable update API, runtime configuration can drift from the workflow state stored in the database.

- <code>thread_id</code>: thread to update.
- <code>metadata</code>: whole-object replacement; <code>None</code> clears metadata and omission preserves it.
- <code>memory_extraction_config</code>: partial grouped update. Provided fields are persisted and omitted fields retain their resolved stored behavior.
- <code>llm</code>: optional live-handle override and is not persisted.
- Ownership identifiers are immutable.
- Inline extraction parameters still exist for compatibility but are deprecated in favor of <code>MemoryExtractionConfig</code>.

The returned <code>OracleThread</code> reflects the persisted configuration, so reassign the handle.


In [ ]:
thread = memory.update_thread(
    THREAD_ID,
    metadata={
        "tenant_id": TENANT_ID,
        "case_id": CASE_ID,
        "status": "replacement-approved",
        "priority": "high",
    },
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_custom_instructions=(
            EXTRACTION_INSTRUCTIONS
            + "\nTreat confirmed replacement approvals as durable facts."
        )
    ),
)

print("Thread status updated:", thread.metadata)


## Demonstrate TTL refresh and explicit deletion

The lifecycle sequence below creates a temporary operational memory.

- Initial <code>ttl_days=7</code> counts from DB creation time.
- <code>update_memory_async(..., ttl_days=30)</code> refreshes expiration.
- Omitting <code>content</code> preserves text.
- <code>delete_memory()</code> returns <code>0</code> or <code>1</code>.
- Thread-scoped APIs only affect memory-like rows owned by this exact thread.
- Expired records are excluded from high-level thread/client operations; the managed purge job later removes physical rows.

Use TTL for retention policy and explicit delete for immediate logical removal.


In [ ]:
temporary_memory_id = await thread.add_memory_async(
    "Temporary courier callback is scheduled for tomorrow.",
    memory_type="memory",
    metadata={**CASE_METADATA, "operational": True},
    ttl_days=7,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

print("Temporary memory:", temporary_memory_id)


Refreshing TTL without providing <code>content</code> preserves the stored content. Providing <code>ttl_anchor</code> without <code>ttl_days</code> would use the schema default duration. Metadata is omitted here so it is preserved rather than replaced.


In [ ]:
await thread.update_memory_async(
    temporary_memory_id,
    ttl_days=30,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

deleted_count = thread.delete_memory(temporary_memory_id)
print("Deleted temporary memories:", deleted_count)


## Prove <code>exact_user_match</code> prevents cross-user leakage

At the high-level client API, searches require an explicit user scope and exact user matching. The example writes a highly relevant record for another customer under the same agent and then searches as the active customer.

- <code>user_id</code> is the requested owner.
- <code>exact_user_match=True</code> requires exact equality.
- <code>exact_agent_match=True</code> also fixes the agent dimension.
- Metadata can add tenant boundaries, but ownership IDs should remain the primary isolation mechanism.
- At lower-level store APIs, a false exact flag means that dimension is unconstrained; use those APIs only behind trusted authorization code.


In [ ]:
OTHER_USER_ID = f"other-customer-{RUN_ID}"
memory.add_user(
    OTHER_USER_ID,
    "A different customer used for an isolation check.",
    metadata={"tenant_id": "other-tenant"},
)

other_memory_id = await memory.add_memory_async(
    "Platinum VIP customer has a damaged order requiring immediate replacement.",
    memory_type="fact",
    user_id=OTHER_USER_ID,
    agent_id=AGENT_ID,
    metadata={"tenant_id": "other-tenant"},
)

primary_user_hits = await memory.search_async(
    "platinum VIP damaged order",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    exact_user_match=True,
    exact_agent_match=True,
    max_results=10,
    record_types=["fact"],
)

assert all(hit.record.id != other_memory_id for hit in primary_user_hits)
print("Cross-user record excluded:", other_memory_id)


## Update the logical policy and its chunk index

The lower-level <code>store.update()</code> supports advanced reindexing. Updating the stored content and its retrieval chunks together is important for policy, contract, and knowledge-base systems: stale chunks can otherwise continue retrieving language that no longer appears in the authoritative document.

- <code>record_type</code> and <code>record_id</code>: select the logical record.
- <code>text</code>: complete replacement logical content.
- <code>index_text</code>: string for store-managed chunking, list for caller-owned chunks, <code>None</code> to clear retrieval rows, or omitted to derive from <code>text</code>.
- <code>embedding</code>: optional precomputed vector or aligned chunk vectors; omitted so Oracle-managed text/hybrid indexing remains authoritative.
- <code>metadata</code>: whole-object replacement.
- TTL fields are omitted, so current expiration is preserved.

With <code>ON_COMMIT</code>, the amended escalation phrase becomes searchable immediately after the update commits.


In [ ]:
updated_policy_chunks = policy_chunks + [
    (
        "Priority handling: when a confirmed accessibility need exists, "
        "expedite the replacement and document the accommodation."
    )
]

updated_count = store.update(
    "guideline",
    POLICY_ID,
    text="\n\n".join(updated_policy_chunks),
    index_text=updated_policy_chunks,
    metadata={
        "tenant_id": TENANT_ID,
        "source": "support-policy",
        "policy_code": "RET-14D",
        "version": 4,
    },
)

print("Updated policy rows:", updated_count)


### Verify chunk reconciliation after the update

Even if several chunks match the query, search returns the logical policy record once. The returned <code>content</code> is the complete updated policy, not only the matching chunk.


In [ ]:
updated_policy_hits = await memory.search_async(
    "expedite a replacement for an accessibility accommodation",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    exact_user_match=True,
    exact_agent_match=True,
    max_results=5,
    record_types=["guideline"],
    metadata_filter={"policy_code": "RET-14D", "version": 4},
)

show_hits(updated_policy_hits)
assert any(hit.record.id == POLICY_ID for hit in updated_policy_hits)


## Compare transcript summary with a context card

Because this thread always has an LLM, <code>get_summary_async()</code> asks the configured model to synthesize the selected transcript.

- <code>except_last=2</code>: excludes the two most recent raw messages from the summary input.
- <code>token_budget=300</code>: soft output budget. Positive values bound output; non-positive values disable the bound.
- The result is an <code>OracleSummary</code>; read <code>.content</code>.
- It summarizes message content only and does not retrieve durable facts or guidelines.

Use a context card when the agent needs retrieval-aware context; use a summary when transcript compression alone is enough.


In [ ]:
thread_summary = await thread.get_summary_async(
    except_last=2,
    token_budget=300,
)

print(thread_summary.content)


## Tear down this run's demo data

Every record this notebook created is scoped by `RUN_ID`, so this step removes
only what this run wrote and leaves the shared `OAMPDEMO` schema in place for
reuse. Deletes cascade to the owned messages, memories, and chunk-index rows,
which keeps the store clean and the notebook safely re-runnable. Comment this
cell out if you want to inspect the data in the database after a run.

In [ ]:
# Drain any in-flight background extraction before deleting so nothing is
# re-created after teardown. The thread still exists at this point.
await thread.wait_for_memory_extraction_async(timeout=120)

teardown_counts = {}


def _safe_delete(label, delete_call):
    try:
        teardown_counts[label] = delete_call()
    except Exception as exc:  # already removed on a re-run, or nothing to delete
        teardown_counts[label] = f"skipped ({type(exc).__name__})"


_safe_delete("guideline_policy", lambda: store.delete("guideline", POLICY_ID, cascade=True))
_safe_delete("thread", lambda: memory.delete_thread(THREAD_ID))
_safe_delete("agent", lambda: memory.delete_agent(AGENT_ID, cascade=True))
_safe_delete("primary_user", lambda: memory.delete_user(USER_ID, cascade=True))
_safe_delete("other_user", lambda: memory.delete_user(OTHER_USER_ID, cascade=True))

for label, count in teardown_counts.items():
    print(f"{label}: {count}")
print("This run's demo data removed; OAMPDEMO schema preserved for reuse.")

## Close background work and the pool

<code>close_async(timeout)</code> stops accepting new background extraction work and waits for already accepted work up to the supplied timeout. It is idempotent.

The connection pool was created by the application, so the application closes it after the memory component has drained. Do not close the pool first while background extraction can still use it.


In [ ]:
await memory.close_async(timeout=120)
db_pool.close()

print("Memory component and Oracle pool closed cleanly.")


## What this architecture achieves

The completed flow uses one continuous support case to demonstrate how the capabilities reinforce each other:

1. **Low request-path latency:** raw messages persist before background extraction finishes.
2. **Higher-quality memory:** custom instructions and typed records keep durable support information focused.
3. **Flexible retrieval:** vector search handles exact policy codes and natural-language paraphrases, and the in-database route adds hybrid keyword fusion.
4. **Deterministic boundaries:** exact scope matching and metadata filters constrain candidates before relevance ranking.
5. **Bounded prompts:** context cards replace full Agents SDK transcript replay while retaining summary, recent turns, and retrieved memory.
6. **Correctable state:** stable IDs and update APIs prevent contradictory append-only memories.
7. **Governed lifecycle:** TTL, explicit updates, deletion, and component shutdown make operational behavior explicit.
8. **Pluggable embeddings:** a hosted OpenAI model by default, or an in-database ONNX model and managed hybrid index that keep embedding and retrieval inside Oracle AI Database.

For production, treat schema creation/upgrades as migrations, use <code>REQUIRE_EXISTING</code> during normal startup, size the pool for concurrent extraction and context retrieval, monitor background queue pressure and extraction failures, and keep authorization checks in application code around every memory tool.
